# SIMD编程概述与Ascend C编程接口概述

上一节通过Hello World示例展示了核函数从编写到调用的过程。本节进一步介绍AI Core如何并行执行计算，以及开发者可以使用哪些Ascend C接口表达计算、搬运和同步逻辑。

本节内容分为两个部分：

- **SIMD编程概述**：理解多核SPMD与单核SIMD、AI Core硬件组件、算子分类及通用开发步骤。
- **Ascend C编程接口概述**：了解语言扩展层C API、基础API、高阶API和Utils API，以及它们与TPipe/TQue框架的关系。


## 1. SIMD编程概述

### 1.1 多核SPMD与单核SIMD

**SIMD（Single Instruction Multiple Data，单指令多数据）**表示一条指令对多组数据执行同一种运算。例如，矢量加法可以对一批输入元素同时执行加法，而无需为每一个元素分别编写一条标量加法语句。

AI Core算子的并行计算可以从两个层面理解：

| 层面 | 编程模型 | 工作方式 |
| --- | --- | --- |
| 多个AI Core之间 | SPMD（Single Program Multiple Data，单程序多数据） | 每个Block执行同一份核函数代码，根据Block索引处理不同的数据分片。 |
| 单个AI Core内部 | SIMD（Single Instruction Multiple Data，单指令多数据） | 由专用计算单元执行指令，批量处理具有相同计算逻辑的数据。 |

在多核任务划分中，每个AI Core可抽象为一个Block，通过`block_idx`区分Block。后续类库示例使用`AscendC::GetBlockIdx()`获取当前Block索引，并据此计算数据偏移。

例如，将16384个元素均匀分配给8个Block，每个Block处理2048个元素。Block 0处理前2048个元素，Block 1处理接下来的2048个元素，以此类推；在每个Block内部，再分批搬入数据并通过矢量指令完成计算。这分别体现了多核SPMD和单核SIMD。


### 1.2 AI Core硬件组件与算子类型

AI Core由控制单元、计算单元和本地存储等组件协同完成计算。可结合前文的[抽象硬件架构](../01_basic_overview/01.04_ascend_c_op_dev_basic_concepts.ipynb)理解以下分工。

| 组件 | 主要职责 |
| --- | --- |
| 标量处理单元（Scalar） | 执行地址偏移计算、分支与循环等控制逻辑，并负责指令调度与发射。 |
| 向量处理单元（Vector） | 批量执行元素级计算、逻辑运算和数据重组等操作。 |
| 矩阵运算单元（Cube） | 执行矩阵乘加等计算，适合规整、计算密集的矩阵运算。 |
| 本地存储（Local Memory） | 保存输入分块与中间结果；Vector计算使用Unified Buffer（UB），Cube计算涉及L1 Buffer、L0C Buffer等存储。 |

根据核心计算逻辑所依赖的计算单元，算子可以分为三类：

- **矢量类算子**：主要使用Vector单元，例如逐元素加法；核函数使用`__vector__`修饰。
- **矩阵类算子**：主要使用Cube单元，例如矩阵乘法；核函数使用`__cube__`修饰。
- **融合类算子**：联合使用Cube与Vector单元，例如矩阵计算后接矢量处理；通过`__mix__(cube, vec)`描述启动的Cube核与Vector核配比。

硬件架构存在分离模式与耦合模式：分离模式下，Cube与Vector各自具有独立的标量控制单元；耦合模式下，两者共享标量控制单元。`__mix__`修饰符仅在分离模式下生效。


### 1.3 显式分层访存与算子开发步骤

AI Core SIMD编程需要开发者显式组织数据流转：将输入从Global Memory（GM，全局内存）搬入Local Memory，在片内完成计算，再将结果写回GM。对于矢量计算，典型数据流为：

<img src="./images/02.03/simd_vector_data_flow.svg" alt="矢量计算数据流：GM输入搬入UB，经Vector计算后将UB结果搬出至GM" width="1000px">

一个SIMD算子的开发通常包含以下四个步骤：

1. **Tiling（分块）设计**：结合数据规模、可用核数和片内存储容量分配任务，确定各Block的数据范围，以及单核内部每次处理的数据量。
2. **数据搬入**：将当前分块从GM搬入UB或L1 Buffer等本地存储。
3. **数据计算**：根据算子逻辑，调用Vector或Cube计算接口处理本地数据。
4. **数据搬出**：将计算结果从本地存储写回GM。

本地存储容量有限，大尺寸张量通常需要循环执行“搬入—计算—搬出”。对于归约、矩阵乘等计算，还需要按算法要求保存和累加中间结果。分块时应处理数据边界，并兼顾负载均衡与接口对齐要求。

不同执行单元可以通过流水线协同工作，使数据搬运与计算发生重叠。下一节将介绍如何用TPipe/TQue组织这些流水任务。


### 1.4 从指针到Tensor的编程接口

Ascend C以标准C/C++语法为基础，提供不同抽象层级的SIMD编程接口。指针编程通过地址直接访问数据；Tensor编程则用`GlobalTensor`和`LocalTensor`分别表示全局内存与本地内存中的数据。

Tensor表示待处理的数据，Tile表示根据存储容量和计算需求划分的数据块。开发者仍需明确分块范围、数据布局、搬运与计算之间的依赖。

| 接口或框架 | 数据表达方式 | 内存与同步管理 |
| --- | --- | --- |
| 语言扩展层SIMD C API | 指针 | 可通过静态数组管理本地内存，由开发者显式管理同步。 |
| C++基础API | Tensor | 可通过`LocalMemoryAllocator`等分配Tensor，由开发者自主管理内存布局与同步。 |
| TPipe/TQue框架 | Tensor与队列 | 通过TPipe管理资源，通过TQue组织任务间的数据传递与同步。 |

TPipe/TQue提供了组织流水线的框架，实际的数据搬运和计算仍需由开发者调用`DataCopy`、`Add`等接口完成。它与基础计算API配合使用，具体用法将在下一节展开。


## 2. Ascend C编程接口概述

Ascend C提供语言扩展层C API和C++类库API，用于表达硬件计算能力、组合常见算法以及完成开发辅助工作。按接口职责，可分为以下四类：

| 类别 | 主要用途 | 典型内容 |
| --- | --- | --- |
| 语言扩展层C API | 面向指针编程，开放芯片编程能力 | 数组内存分配、计算、搬运与同步接口 |
| 基础API | 抽象硬件能力，组合实现算子逻辑 | 标量、矢量、矩阵计算，数据搬运、资源管理和同步控制 |
| 高阶API | 封装常见算法，提高开发效率 | 数学库、Matmul、Softmax等 |
| Utils API | 提供通用辅助能力 | C++标准库、平台信息、RTC、日志和调测接口 |

Ascend C编程接口的整体架构如下图所示：

<img src="./images/02.03/architecture_ascendc.png" alt="Ascend C编程接口架构图" width="1000px">

这一分类描述接口的职责；前文的三种编程方式描述开发者如何组织数据、资源和同步。实际开发时可以组合使用不同类别的接口。

基础API通常兼顾能力完备性与兼容性，但标注为**ISASI（Instruction Set Architecture Special Interface，硬件体系结构相关接口）**的接口不保证跨硬件版本兼容。选用接口时，应查看对应文档中的产品支持范围、数据类型和参数约束。


### 2.1 语言扩展层C API

语言扩展层C API开放芯片编程能力，提供接近C语言的指针编程体验。开发者可以通过数组分配本地内存，使用指针表示计算和搬运的数据，并显式控制指令间的同步。

这种方式适合希望直接管理地址、内存与同步的开发场景。与基于Tensor的C++类库接口相比，其数据表达方式和资源管理方式不同；两者都可以用于实现SIMD算子。


### 2.2 基础API

基础API将硬件能力抽象为可组合的接口。开发者通过计算、搬运、资源管理与同步接口实现自己的算子逻辑。

| 功能类别 | 作用与示例 |
| --- | --- |
| 标量计算 | 调用Scalar单元执行标量计算。 |
| 矢量计算 | 调用Vector单元完成批量数据计算，例如`Add`。 |
| 矩阵计算 | 调用Cube单元完成矩阵计算。 |
| 数据搬运 | 在不同存储位置间传递数据，例如`DataCopy`。 |
| 资源管理 | 分配和回收计算所需内存，例如队列的`AllocTensor`、`FreeTensor`。 |
| 同步控制 | 保证存在依赖的操作按正确顺序执行；框架中可通过`EnQue`、`DeQue`完成任务间通信与同步。 |

不同指令队列可以异步执行，因此“代码书写顺序”不足以替代必要的硬件同步。使用TPipe/TQue框架时，队列接口封装了相应同步处理；采用自主同步方式时，需要开发者明确管理这些依赖。


### 2.3 连续计算API与高维切分API

对于矢量计算基础API，还可以按数据操作方式区分为连续计算API和高维切分API。

**连续计算API**处理Tensor前`n`个连续元素，并将结果连续写入目的Tensor。例如，以下接口表达两个输入Tensor前`n`个元素的逐元素相加：

```cpp
// 示意：参与计算的LocalTensor已分配，输入数据已就绪。
AscendC::Add(dstLocal, src0Local, src1Local, n);
```

**高维切分API**允许进一步控制迭代次数、数据地址间隔和参与计算的元素，适合需要灵活安排数据访问的场景。

| 参数概念 | 含义 |
| --- | --- |
| repeatTime | 指令迭代执行的次数。 |
| dataBlockStride | 同一次迭代中，不同数据块起始地址之间的步长。 |
| repeatStride | 相邻迭代中，同一位置的数据块起始地址之间的步长。 |
| mask | 控制每次迭代中实际参与计算的元素。 |

这些是通用参数概念，具体API中的参数名、步长单位和取值范围应以对应接口文档为准。例如，双目计算接口会分别设置目的操作数和两个源操作数的步长。

<img src="./images/02.03/characteristics_of_several_computing_methods_of_computing_api.png" alt="连续计算API与高维切分API的特点" width="700px">

#### 掩码操作API

掩码（Mask）用于控制矢量计算中实际参与计算的元素。根据使用场景，可以选择两种工作模式：

- **Normal模式**：调用`SetMaskNorm()`设置。Mask控制**单次迭代**内参与计算的元素，适合需要精确控制每次迭代数据的场景。
- **Counter模式**：调用`SetMaskCount()`设置。Mask表示**整个矢量计算**参与计算的元素个数，接口可以自动推断迭代次数，适合处理非对齐尾块。

Mask既可以作为计算API参数传入，也可以通过外部API配置。需要多次复用同一个Mask时，可以调用`SetVectorMask`设置Mask，并将计算API的`isSetMask`模板参数设为`false`，计算API中的Mask参数使用`MASK_PLACEHOLDER`占位。使用完毕后调用`ResetMask()`恢复默认Mask；手动使用Counter模式后调用`SetMaskNorm()`恢复Normal模式。

下面示例使用Counter模式配置128个元素，并通过外部Mask配置重复调用高维切分计算API：

```cpp
AscendC::SetMaskCount();
AscendC::SetVectorMask<half, AscendC::MaskMode::COUNTER>(128);
AscendC::Add<half, false>(dstLocal, src0Local, src1Local,
    AscendC::MASK_PLACEHOLDER, 1, {1, 1, 1, 8, 8, 8});
AscendC::SetMaskNorm();
AscendC::ResetMask();
```

如果每次调用的Mask不同，也可以直接将Mask或`calCount`作为计算API参数传入。Normal模式下还需要按`repeatTime`处理尾块；Counter模式下应根据实际数据量设置Mask。具体参数类型、支持的接口和取值范围以对应API文档为准。

#### 归约API

归约将一组数据压缩为单个值或更小的数据集合，适合求最大值、最小值、求和等场景。常用归约API按处理范围分为：

| API | 处理范围 | 结果示例 |
| --- | --- | --- |
| `ReduceMax`、`ReduceMin`、`ReduceSum` | 所有输入数据 | 全局最大值、最小值或总和；部分接口还可返回索引。 |
| `ReduceRepeat` | 每个 repeat 内的数据 | 每个 repeat 的最大值、最小值或总和。 |
| `ReduceDataBlock` | 每个 DataBlock 内的数据 | 每个 DataBlock 的最大值、最小值或总和。 |
| `ReducePairElem` | 相邻两个元素 | 当前主要支持求和，例如`(a1,a2,a3,a4)`变为`(a1+a2,a3+a4)`。 |

归约API同样支持Mask、`repeatTime`和地址步长，但归约后目的Tensor长度会变短，目的操作数的`repeatStride`和`dataBlockStride`约束可能与普通矢量计算不同。`ReduceMax/ReduceMin/ReduceSum`的目的操作数不支持配置这些步长；使用前应根据具体API说明准备输入输出Tensor和边界。

以下为归约接口的示意流程，实际函数签名和模板参数请以安装版本的API文档为准：

```cpp
// srcLocal中保存待归约数据，dstLocal保存归约结果
// 选择ReduceType后，对每个repeat执行对应的归约操作
AscendC::ReduceRepeat<AscendC::ReduceType::SUM>(
    dstLocal, srcLocal, mask, repeatTime, srcRepStride, dstRepStride);
```

归约计算的Mask配置方式与基础矢量API一致：可以通过接口传参，也可以先调用`SetVectorMask`进行外部配置。处理大张量时，还需结合Tiling分块并在需要时累加多个分块的中间结果。更多掩码和归约API示例请参考本节的课外参考。


### 2.4 高阶API

高阶API基于单核对常见算法进行抽象与封装，一般由多种基础API组合实现，覆盖数学计算、矩阵计算、激活函数等场景，例如Matmul、Softmax。

使用基础API实现矩阵乘法时，需要组织数据搬运、格式转换、数据切分和计算等步骤；高阶API封装其中的实现细节，便于开发者更快完成算法功能。

<img src="./images/02.03/high_level_api.png" alt="基础API与高阶API实现矩阵乘法的流程对比" width="700px">

使用高阶API时，仍需遵循接口对输入输出、Tiling参数和临时空间的要求。对于配套Host Tiling接口的调用，需要链接`libtiling_api.a`；应使用文档明确公开的接口，而非直接依赖头文件中的内部实现。


### 2.5 Utils API（公共辅助函数）

Utils API提供通用工具，辅助完成算子开发、运行与调试。

- **C++标准库API**：提供算法、数学函数和容器等功能。
- **平台信息获取API**：查询硬件平台信息，例如可用核数，为任务划分提供依据。
- **RTC API**：通过`aclrtc`接口在运行时将中间代码动态编译为目标机器码。
- **log API**：提供Host侧日志功能，例如在`TilingFunc`中使用`ASC_CPU_LOG_XXX`输出信息。
- **调测接口**：面向调试场景提供辅助能力。


## 3. 课后练习

以下5道单选题覆盖本节的 SIMD 编程与 Ascend C 编程接口内容，每题只有一个正确答案。

1. （单选题）将16384个元素均匀分配给8个Block，每个Block运行同一份核函数并处理2048个元素；核内使用矢量指令批量计算。外层与内层分别对应哪种并行模型？

    A. 外层SIMD，内层SPMD

    B. 外层SPMD，内层SIMD

    C. 外层串行，内层SPMD

    D. 外层与内层都是串行

2. （单选题）需要让一次矢量计算只处理指定数量的元素，并自动推断迭代次数、减少尾块处理逻辑，应优先使用哪种 Mask 模式？

    A. Normal 模式

    B. Counter 模式

    C. 关闭 Mask

    D. 仅设置 repeatStride

3. （单选题）以下哪项正确描述了 `ReduceRepeat`？

    A. 只负责将数据从 GM 搬入 UB

    B. 对每个 repeat 内的数据做归约，并输出每个 repeat 的更小结果集合

    C. 只能处理相邻两个元素的求和

    D. 不支持 Mask 和 repeatTime

4. （单选题）希望通过指针表达数据，并使用数组分配本地内存，应重点了解哪类接口？

    A. Utils API

    B. 高阶API

    C. 日志API

    D. 语言扩展层C API

5. （单选题）当同一个 Mask 需要被多个矢量计算 API 重复使用时，推荐的方式是什么？

    A. 每次调用前都重新分配 LocalTensor

    B. 调用 `SetVectorMask` 外部配置 Mask，将 `isSetMask` 设为 `false`，使用后调用 `ResetMask`

    C. 只调用 `SetMaskNorm`，不设置具体 Mask

    D. 将 Mask 写入 Global Memory 后由 Vector 单元自动读取

**完成自测后，执行以下代码查看答案与解析。**


In [ ]:
!cat ./answer/02.03/02.03_answer.txt

## 4. 课外参考

本节内容参考 Ascend C DevKit 的编程接口文档。掩码操作、归约计算、基础 API 的参数约束及更多示例，可继续阅读：

- [Ascend C Library API 文档（9.2.0）](https://gitcode.com/cann/asc-devkit/tree/9.2.0/docs/zh/guide/programming_guide/library_api)
- [如何使用掩码操作 API](https://gitcode.com/cann/asc-devkit/blob/9.2.0/docs/zh/guide/programming_guide/library_api/basic_api/quick_reference/how_to_use_mask_api.md)
- [如何使用归约计算 API](https://gitcode.com/cann/asc-devkit/blob/9.2.0/docs/zh/guide/programming_guide/library_api/basic_api/quick_reference/how_to_use_reduction_api.md)

使用时请结合本地安装的 CANN 版本核对接口签名、支持的数据类型、硬件版本和参数范围。
